# Aula 8 — Redes Convolucionais

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

O Keras já vem instalado no Google Colab, e as fotos vêm dentro dele.
Nenhuma instalação e nenhum download manual.

Aviso sobre o tempo: treinar as redes desta aula leva alguns minutos no
Colab sem placa de vídeo. É normal. Rode a célula e espere.

## Parte A: Demonstração

### As fotos: 70.000 peças de roupa

O Fashion-MNIST vem dentro do Keras. São 60.000 fotos para treinar e
10.000 para testar, todas de 28 por 28 pixels em tons de cinza.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras

(fotos_treino, rotulos_treino), (fotos_teste, rotulos_teste) = \
    keras.datasets.fashion_mnist.load_data()

CLASSES = ["camiseta", "calça", "pulôver", "vestido", "casaco",
           "sandália", "camisa", "tênis", "bolsa", "bota"]

print(f"Treino: {fotos_treino.shape}")
print(f"Teste:  {fotos_teste.shape}")
print(f"Valores dos pixels: de {fotos_treino.min()} a {fotos_treino.max()}")

In [ ]:
plt.figure(figsize=(12, 5))
for indice in range(10):
    primeira = np.where(rotulos_treino == indice)[0][0]
    plt.subplot(2, 5, indice + 1)
    plt.imshow(fotos_treino[primeira], cmap="gray_r")
    plt.title(CLASSES[indice])
    plt.axis("off")
plt.show()

### Uma foto é uma tabela de números

Olhe um pedaço de 8 por 8 de uma foto, como o computador vê.

In [ ]:
foto = fotos_treino[0]
print(f"A foto é uma tabela {foto.shape}, ou seja, {foto.size} números.")
print(f"Esta é uma foto de: {CLASSES[rotulos_treino[0]]}")
print()
print(foto[10:18, 8:16])

### Normalizar: de 0 a 255 para 0 a 1

Como na Aula 7, a rede quer números pequenos. Aqui é ainda mais simples:
todo pixel está entre 0 e 255, então basta dividir.

$$x_{\text{normalizado}} = \frac{x}{255}$$

In [ ]:
fotos_treino = fotos_treino / 255.0
fotos_teste = fotos_teste / 255.0

print(f"Agora os pixels vão de {fotos_treino.min():.1f} a {fotos_treino.max():.1f}")

### O filtro, na mão

Antes de deixar o Keras fazer, vamos fazer uma convolução com dois `for`.
Cada número da saída é a soma de nove multiplicações:

$$s_{i,j} = \sum_{a=0}^{2}\sum_{b=0}^{2} k_{a,b} \cdot x_{i+a,\,j+b}$$

In [ ]:
def aplicar_filtro(imagem, filtro):
    altura, largura = imagem.shape
    saida = np.zeros((altura - 2, largura - 2))
    for i in range(altura - 2):
        for j in range(largura - 2):
            pedaco = imagem[i:i + 3, j:j + 3]
            saida[i, j] = np.sum(pedaco * filtro)
    return np.abs(saida)

borda_vertical = np.array([[-1, 0, 1],
                           [-1, 0, 1],
                           [-1, 0, 1]])

camiseta = fotos_treino[np.where(rotulos_treino == 0)[0][0]]

plt.figure(figsize=(11, 4))
plt.subplot(1, 3, 1)
plt.imshow(camiseta, cmap="gray_r")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(aplicar_filtro(camiseta, borda_vertical), cmap="gray_r")
plt.title("Bordas verticais")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(aplicar_filtro(camiseta, borda_vertical.T), cmap="gray_r")
plt.title("Bordas horizontais")
plt.axis("off")
plt.show()

### A rede densa da Aula 7, como régua

`Flatten` enfileira os 784 pixels. Depois disso, a rede não sabe mais
quem era vizinho de quem.

In [ ]:
keras.utils.set_random_seed(42)

rede_densa = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])

rede_densa.compile(optimizer="adam",
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])
rede_densa.summary()

In [ ]:
# Leva cerca de um minuto no Colab. Espere a célula terminar.
rede_densa.fit(fotos_treino, rotulos_treino, epochs=8, batch_size=64, verbose=0)

acuracia_densa = rede_densa.evaluate(fotos_teste, rotulos_teste, verbose=0)[1]
print(f"Rede densa: {acuracia_densa:.1%} de acerto em 10.000 fotos novas")
print(f"Pesos: {rede_densa.count_params():,}")

### A rede convolucional

O `1` no fim de `(28, 28, 1)` é o canal. Foto em cinza tem um número por
pixel; foto colorida teria três.

In [ ]:
fotos_treino_canal = fotos_treino.reshape(-1, 28, 28, 1)
fotos_teste_canal = fotos_teste.reshape(-1, 28, 28, 1)

keras.utils.set_random_seed(42)

rede_conv = keras.Sequential([
    keras.layers.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(16, 3, activation="relu"),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPooling2D(2),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])

rede_conv.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
rede_conv.summary()

In [ ]:
# Esta é a mais demorada: alguns minutos no Colab.
rede_conv.fit(fotos_treino_canal, rotulos_treino, epochs=8, batch_size=64,
              verbose=0)

acuracia_conv = rede_conv.evaluate(fotos_teste_canal, rotulos_teste, verbose=0)[1]

print(f"Densa:         {acuracia_densa:.1%} com {rede_densa.count_params():,} pesos")
print(f"Convolucional: {acuracia_conv:.1%} com {rede_conv.count_params():,} pesos")

### O teste do deslocamento

Empurre as fotos de teste alguns pixels para o lado. A peça é a mesma, só
mudou de lugar. Quem aguenta?

In [ ]:
for passo in [0, 2, 4]:
    movidas = np.roll(fotos_teste, passo, axis=2)
    d = rede_densa.evaluate(movidas, rotulos_teste, verbose=0)[1]
    c = rede_conv.evaluate(movidas.reshape(-1, 28, 28, 1), rotulos_teste,
                           verbose=0)[1]
    print(f"{passo} pixels: densa {d:.1%}, convolucional {c:.1%}")

### Onde ela erra: a matriz de confusão

A linha diz o que a peça era. A coluna diz o que o modelo respondeu.

In [ ]:
from sklearn.metrics import confusion_matrix

respostas = rede_conv.predict(fotos_teste_canal, verbose=0).argmax(axis=1)
matriz = confusion_matrix(rotulos_teste, respostas)

plt.figure(figsize=(8, 7))
plt.imshow(matriz, cmap="Greys", alpha=0.6)
for linha in range(10):
    for coluna in range(10):
        if matriz[linha, coluna] > 0:
            plt.text(coluna, linha, matriz[linha, coluna],
                     ha="center", va="center", fontsize=9)
plt.xticks(range(10), CLASSES, rotation=45, ha="right")
plt.yticks(range(10), CLASSES)
plt.xlabel("O que o modelo respondeu")
plt.ylabel("O que era de verdade")
plt.show()

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conhecendo as fotos

Rode a célula e observe: o conjunto está equilibrado entre as dez
categorias?

In [ ]:
for indice, nome in enumerate(CLASSES):
    quantas = (rotulos_treino == indice).sum()
    print(f"{nome:<10} {quantas} fotos")

In [ ]:
if len(rotulos_treino) == 60000:
    print("✅ 60.000 fotos de treino, 6.000 por categoria. Perfeitamente equilibrado.")
else:
    print("❌ Confira se você rodou a célula que carrega as fotos.")

### Exercício 2: acertando o formato

A `Conv2D` quer o canal no fim. Crie `minhas_fotos` com formato
`(60000, 28, 28, 1)`, a partir de `fotos_treino`.

Dica: `reshape(-1, 28, 28, 1)`. O `-1` quer dizer "descubra quantas são".

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(minhas_fotos.shape)

In [ ]:
if minhas_fotos.shape == (60000, 28, 28, 1):
    print("✅ Formato certo: 60.000 fotos de 28 por 28, com 1 canal.")
else:
    print(f"❌ Esperava (60000, 28, 28, 1) e veio {minhas_fotos.shape}.")

### Exercício 3: um filtro na mão

Complete o filtro que acha bordas **horizontais**. Ele é o de bordas
verticais deitado: a primeira linha com `-1`, a do meio com `0` e a de
baixo com `1`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
bota = fotos_treino[np.where(rotulos_treino == 9)[0][0]]

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(bota, cmap="gray_r")
plt.title("A bota")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(aplicar_filtro(bota, meu_filtro), cmap="gray_r")
plt.title("Depois do seu filtro")
plt.axis("off")
plt.show()

In [ ]:
if meu_filtro.shape == (3, 3) and meu_filtro.sum() == 0 and meu_filtro[1].sum() == 0:
    print("✅ Filtro de bordas horizontais: a sola e o cano da bota ficaram marcados.")
else:
    print("❌ Confira: três linhas, com -1 em cima, 0 no meio e 1 embaixo.")

### Exercício 4: montando a sua rede

Monte `minha_rede` com **uma** camada de convolução de 8 filtros 3×3 com
ReLU, um `MaxPooling2D(2)`, um `Flatten` e a saída `Dense(10)` com
softmax. Depois compile com `optimizer="adam"` e
`loss="sparse_categorical_crossentropy"`.

In [ ]:
keras.utils.set_random_seed(7)
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_rede.summary()

In [ ]:
if minha_rede.count_params() == 13610:
    print("✅ 13.610 pesos: 80 na convolução e 13.530 na saída.")
    print("   Repare como a parte que enxerga é a barata.")
else:
    print(f"❌ Esperava 13.610 pesos e vieram {minha_rede.count_params()}.")

### Exercício 5: treinando

Treine `minha_rede` por **5** épocas, com `batch_size=64` e
`validation_split=0.1`. Guarde o resultado em `meu_historico`.

Isso leva alguns minutos no Colab. Rode e espere.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
minha_acuracia = minha_rede.evaluate(fotos_teste.reshape(-1, 28, 28, 1),
                                     rotulos_teste, verbose=0)[1]
print(f"Sua rede:      {minha_acuracia:.1%}")
print(f"Rede densa:    {acuracia_densa:.1%}")
print(f"Rede da aula:  {acuracia_conv:.1%}")

In [ ]:
if minha_acuracia > 0.85:
    print("✅ Com uma só camada de convolução você já passou dos 85%.")
else:
    print("❌ Confira se você treinou com minhas_fotos, que tem o canal.")

### Exercício 6: olhando os erros

Rode e observe as seis primeiras fotos que a sua rede errou. Você teria
acertado?

In [ ]:
minhas_respostas = minha_rede.predict(fotos_teste.reshape(-1, 28, 28, 1),
                                      verbose=0).argmax(axis=1)
erros = np.where(minhas_respostas != rotulos_teste)[0]

plt.figure(figsize=(12, 4))
for posicao, indice in enumerate(erros[:6]):
    plt.subplot(1, 6, posicao + 1)
    plt.imshow(fotos_teste[indice], cmap="gray_r")
    plt.title(f"era {CLASSES[rotulos_teste[indice]]}\ndisse {CLASSES[minhas_respostas[indice]]}",
              fontsize=9)
    plt.axis("off")
plt.show()

In [ ]:
print(f"A sua rede errou {len(erros)} das 10.000 fotos de teste.")
print("Converse com um colega: quantas dessas seis você também erraria?")

### Exercício 7: desafio, a pior categoria

Descubra qual categoria a sua rede mais erra. Calcule o acerto de cada
uma e guarde o nome da pior em `pior_categoria`.

Dica: para a categoria `i`, o acerto é a quantidade de fotos dessa
categoria que o modelo respondeu `i`, dividida pelo total de fotos dela.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
# SEU CODIGO AQUI

In [ ]:
for indice, nome in enumerate(CLASSES):
    print(f"{nome:<10} {acertos[indice]:.1%}")
print()
print(f"A pior é: {pior_categoria}")

In [ ]:
if pior_categoria == "camisa":
    print("✅ Camisa. Ela se parece com camiseta, pulôver e casaco em 28×28 sem cor.")
    print("   Uma pessoa erraria as mesmas.")
else:
    print(f"Sua rede achou {pior_categoria} a pior. Na rede da aula é a camisa.")
    print("Redes treinam com números aleatórios, então isso pode variar um pouco.")

Agora, em texto: o cliente do brechó pergunta se pode confiar no seu
modelo. Responda em três frases, usando o que você descobriu no exercício
7. Edite esta célula (duplo clique nela) e escreva sua resposta no lugar
deste parágrafo.